
# Аналитика авторов: потенциал соцкоммерции и оптимальные форматы
Визуализации построены в фирменных цветах ВКонтакте, подписи — на русском. Итоговые выводы включают ответы на ключевые вопросы:
1. Какие авторы обладают наибольшим контентным потенциалом для соцкоммерции и какие сегменты приоритетны?
2. Какие виды и типы контента перспективны для каждого сегмента авторов и товарных категорий?


In [1]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from pathlib import Path

px.defaults.template = 'plotly_white'
px.defaults.color_discrete_sequence = ['#2787F5', '#4A76A8', '#50C878', '#5A9CF5', '#91C4FF']
pio.renderers.default = 'plotly_mimetype'

# Пути и загрузка
ZIP_PATH = Path('df_authors (1).csv.zip')
CSV_NAME = 'df_authors (1).csv'

with ZIP_PATH.open('rb') as f:
    import zipfile
    with zipfile.ZipFile(f) as zf:
        with zf.open(CSV_NAME) as csvfile:
            df_raw = pd.read_csv(csvfile)

# Переименуем столбцы на русский для удобства восприятия
col_map = {
    'hash_author_id': 'ID автора',
    'segment_code': 'Сегмент',
    'size': 'Размер аудитории',
    'category': 'Категория',
    'original': 'Оригинальность',
    'quality': 'Качество',
    'expert': 'Экспертность',
    'posts_30d': 'Посты 30д',
    'posts_views_30d': 'Просмотры постов 30д',
    'posts_reach_30d': 'Охват постов 30д',
    'posts_likes_30d': 'Лайки постов 30д',
    'posts_comments_30d': 'Комментарии постов 30д',
    'posts_reposts_30d': 'Репосты постов 30д',
    'clips_30d': 'Клипы 30д',
    'clips_views_30d': 'Просмотры клипов 30д',
    'clips_reach_30d': 'Охват клипов 30д',
    'clips_likes_30d': 'Лайки клипов 30д',
    'clips_comments_30d': 'Комментарии клипов 30д',
    'clips_reposts_30d': 'Репосты клипов 30д',
}

df = df_raw.rename(columns=col_map)

# Производные метрики
for col in ['Охват постов 30д', 'Просмотры постов 30д', 'Лайки постов 30д', 'Комментарии постов 30д', 'Репосты постов 30д',
            'Охват клипов 30д', 'Просмотры клипов 30д', 'Лайки клипов 30д', 'Комментарии клипов 30д', 'Репосты клипов 30д',
            'Посты 30д', 'Клипы 30д', 'Размер аудитории']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Вовлеченность и охват
df['Вовлеченность постов'] = df['Лайки постов 30д'] + df['Комментарии постов 30д'] + df['Репосты постов 30д']
df['Вовлеченность клипов'] = df['Лайки клипов 30д'] + df['Комментарии клипов 30д'] + df['Репосты клипов 30д']
df['Охват суммарный 30д'] = df[['Охват постов 30д', 'Охват клипов 30д']].sum(axis=1)
df['Объем контента 30д'] = df[['Посты 30д', 'Клипы 30д']].sum(axis=1)

# Engagement rate с защитой от деления на ноль
for reach_col, eng_col, target_col in [
    ('Охват постов 30д', 'Вовлеченность постов', 'ER постов'),
    ('Охват клипов 30д', 'Вовлеченность клипов', 'ER клипов'),
    ('Охват суммарный 30д', ['Вовлеченность постов', 'Вовлеченность клипов'], 'ER суммарный'),
]:
    if isinstance(eng_col, list):
        eng = df[eng_col].sum(axis=1)
    else:
        eng = df[eng_col]
    df[target_col] = np.where(df[reach_col] > 0, eng / df[reach_col], 0)

# Интегральный потенциал: охват + вовлеченность + продуктивность
reach_rank = df['Охват суммарный 30д'].rank(pct=True)
eng_rank = df['ER суммарный'].rank(pct=True)
volume_rank = df['Объем контента 30д'].rank(pct=True)
df['Интегральный потенциал'] = (0.45 * reach_rank + 0.35 * eng_rank + 0.20 * volume_rank)

# Краткий вид для вывода
print('Размер датасета:', df.shape)
df.head(3)


Размер датасета: (305341, 27)


,ID автора,Сегмент,Размер аудитории,Категория,Оригинальность,Качество,Экспертность,Посты 30д,Просмотры постов 30д,Охват постов 30д,...,Комментарии клипов 30д,Репосты клипов 30д,Вовлеченность постов,Вовлеченность клипов,Охват суммарный 30д,Объем контента 30д,ER постов,ER клипов,ER суммарный,Интегральный потенциал
0,6B86B273FF34FCE19D6B804EFF5A3F5747ADA4EAA22F1D...,2,213021.0,SPORT,aggregator,like,pugc,286.0,9863513.0,6310430.0,...,0.0,0.0,68260.0,0.0,6310430.0,286.0,0.010817,0.000000,0.010817,0.722376
1,D4735E3A265E16EEE03F59718B9B5D03019C07D8B6C51F...,2,40895.0,COMMUNITY,aggregator,like,pugc,444.0,1481121.0,910920.0,...,0.0,0.0,54497.0,3.0,911965.0,446.0,0.059826,0.002871,0.059761,0.863769
2,4E07408562BEDB8B60CE05C1DECFE3AD16B72230967DE0...,2,228547.0,SPORT,aggregator,like,pugc,46.0,2843760.0,1761202.0,...,70.0,0.0,20919.0,6979.0,2212390.0,64.0,0.011878,0.015468,0.012610,0.634698



## Качество данных
Оценим пропуски и разброс ключевых числовых метрик.


In [2]:
missing_pct = df.isna().mean().sort_values(ascending=False) * 100
missing_top = missing_pct.head(10).reset_index()
missing_top.columns = ['Показатель', 'Доля пропусков, %']

fig_missing = px.bar(
    missing_top,
    x='Показатель', y='Доля пропусков, %',
    color='Доля пропусков, %',
    title='Топ-10 полей по доле пропусков',
    labels={'Показатель': 'Поле'},
)
fig_missing.update_layout(xaxis_tickangle=-35)

stats = df[['Охват суммарный 30д', 'ER суммарный', 'Объем контента 30д', 'Размер аудитории']].describe().loc[['mean','50%','std','min','max']]

missing_top, stats
display(fig_missing)
display(missing_top)
display(stats)

,Показатель,"Доля пропусков, %"
0,Оригинальность,14.143859
1,Качество,10.340570
2,Экспертность,10.340570
3,Категория,1.193420
4,Посты 30д,0.111351
5,Размер аудитории,0.111351
6,Просмотры постов 30д,0.111351
7,Охват постов 30д,0.111351
8,Лайки постов 30д,0.111351
9,Репосты постов 30д,0.111351


,Охват суммарный 30д,ER суммарный,Объем контента 30д,Размер аудитории
mean,8.097645e+05,0.040558,244.043276,3.564047e+04
50%,7.107400e+04,0.018392,84.000000,5.844000e+03
std,4.597649e+06,0.125204,1309.113365,1.905675e+05
min,0.000000e+00,0.000000,0.000000,8.000000e+00
max,7.301323e+08,14.716981,408290.000000,2.259818e+07



## Сегменты и категории
Смотрим, где сосредоточены авторы и каков их средний потенциал.


In [3]:
segment_summary = (
    df.groupby('Сегмент')
      .agg({
          'ID автора': 'count',
          'Интегральный потенциал': 'mean',
          'Охват суммарный 30д': 'mean',
          'ER суммарный': 'mean'
      })
      .rename(columns={'ID автора': 'Количество авторов'})
      .reset_index()
)

fig_seg = px.bar(
    segment_summary.sort_values('Интегральный потенциал', ascending=False),
    x='Сегмент', y='Интегральный потенциал', color='Количество авторов',
    title='Средний потенциал по сегментам',
    labels={'Сегмент': 'Код сегмента'},
)

category_summary = (
    df.groupby('Категория')
      .agg({
          'ID автора': 'count',
          'Интегральный потенциал': 'mean',
          'ER суммарный': 'mean'
      })
      .rename(columns={'ID автора': 'Количество авторов'})
      .reset_index()
)

fig_cat = px.scatter(
    category_summary,
    x='Количество авторов', y='Интегральный потенциал',
    size='ER суммарный', color='Интегральный потенциал',
    title='Категории: потенциал vs. масштаб',
    hover_name='Категория'
)

segment_summary.head(10), category_summary.sort_values('Интегральный потенциал', ascending=False).head(10)
display(fig_seg)
display(fig_cat)
display(segment_summary.head(10))
display(category_summary.sort_values('Интегральный потенциал', ascending=False).head(10))

,Сегмент,Количество авторов,Интегральный потенциал,Охват суммарный 30д,ER суммарный
0,1,640,0.632967,1.041335e+07,0.016561
1,2,69775,0.557407,1.840021e+06,0.039133
2,3,155791,0.496575,4.617151e+05,0.033357
3,4,48047,0.498841,7.815171e+05,0.061727
4,5,143,0.658176,7.566103e+05,0.063440
5,6,30945,0.386134,8.445517e+04,0.047542


,Категория,Количество авторов,Интегральный потенциал,ER суммарный
0,ANIMALS,5232,0.646174,0.080001
29,ENTERTAINMENT,19960,0.587574,0.039993
27,CULTURE_SOCIETY,32810,0.572378,0.051732
5,COMMUNITY,13143,0.565744,0.055384
17,COMPANY_MASS_MEDIA,5730,0.559335,0.024114
32,FOOD_COOKING,2726,0.552387,0.021122
22,COMPANY_SOCIETY,13897,0.550367,0.038907
30,FAMILY,7046,0.544326,0.040273
10,COMPANY_CULTURE,6204,0.540371,0.046079
11,COMPANY_EDUCATION,18523,0.539675,0.041119



## Какие авторы наиболее перспективны
Отберём топ-15 по интегральному потенциалу и покажем их профиль.


In [4]:
TOP_N = 15
selected_cols = ['ID автора', 'Сегмент', 'Категория', 'Размер аудитории', 'Объем контента 30д', 'Охват суммарный 30д', 'ER суммарный', 'Интегральный потенциал', 'ER постов', 'ER клипов']

top_authors = df.sort_values('Интегральный потенциал', ascending=False).head(TOP_N)[selected_cols].reset_index(drop=True)

fig_top = px.bar(
    top_authors.assign(**{'ID автора': top_authors['ID автора'].str.slice(0, 8)}),
    x='ID автора', y='Интегральный потенциал', color='Сегмент',
    title='Топ-15 авторов по потенциалу',
    hover_data=['Категория', 'Размер аудитории', 'ER суммарный']
)

fig_top.update_layout(xaxis_title='ID автора (усечён)', yaxis_title='Интегральный потенциал')

top_authors
display(fig_top)
display(top_authors)

,ID автора,Сегмент,Категория,Размер аудитории,Объем контента 30д,Охват суммарный 30д,ER суммарный,Интегральный потенциал,ER постов,ER клипов
0,AE600B4ED00AFCDBB84721F4BADB1BC9B05B126A26960B...,4,CULTURE_SOCIETY,264124.0,1086.0,40415180.0,0.193981,0.984350,0.218553,0.192931
1,48BE77E1F3AF493E2529A25E866753F659DC85E2D8FC8C...,3,COMMUNITY,34502.0,1712.0,9942797.0,0.167196,0.979761,0.167196,0.000000
2,FB720ED6D9FC50F60C7936F3CF8BAC24D467EAC2FEA5F9...,4,COMPANY_MASS_MEDIA,713613.0,1480.0,6717799.0,0.231640,0.979477,0.231640,0.000000
3,9AF899BDDDDFD92AE104DD7083B96A5B36A919A3812D4B...,4,CULTURE_SOCIETY,14033.0,5466.0,6645543.0,0.165067,0.978338,0.568064,0.156323
4,15B6C239A00C93D8A8938EFEBE629C22525A41A92084B0...,2,BUSINESS,402124.0,1494.0,85837728.0,0.117603,0.976693,0.045505,0.120263
5,A3C830E1EFC8694EB7EF1249587C3B19EA8D1205EDFFF2...,2,CULTURE_SOCIETY,159679.0,1348.0,9667262.0,0.151497,0.976185,0.150972,0.191838
6,ADEB3A47CA0909906CD03949AFA61F7326D2831EA4129E...,2,COMMUNITY,24171.0,4354.0,6884719.0,0.146500,0.976119,0.156720,0.146231
7,A08476F4E6B842062E80E049ED2A51B76913A447C220C2...,2,COMMUNITY,26614.0,2052.0,6441912.0,0.159351,0.975464,0.158862,0.244676
8,2A6FC8B004F35EFB6CCA1D6C7B1C0E961D21DF48D904DD...,2,COMMUNITY,27919.0,1560.0,6266372.0,0.171625,0.975094,0.168882,0.217956
9,60124FA76DD63D94D50A616B021EE12E62917514769F0B...,3,FAMILY,45709.0,3760.0,6048878.0,0.150158,0.974889,0.150024,0.212427



## Форматы: посты vs клипы
Сравним вовлечённость и охват форматов по категориям, чтобы понять, что работает лучше для соцкоммерции.


In [5]:
format_perf = (
    df.groupby('Категория')
      .agg({
          'ER постов': 'mean',
          'ER клипов': 'mean',
          'Охват постов 30д': 'mean',
          'Охват клипов 30д': 'mean',
          'Интегральный потенциал': 'mean'
      })
      .reset_index()
)

fig_formats = px.scatter(
    format_perf,
    x='ER постов', y='ER клипов',
    size='Интегральный потенциал', color='Интегральный потенциал',
    title='ER постов vs ER клипов по категориям',
    hover_data=['Категория', 'Охват постов 30д', 'Охват клипов 30д']
)

fig_reach = px.bar(
    format_perf.melt(id_vars='Категория', value_vars=['Охват постов 30д', 'Охват клипов 30д'], var_name='Формат', value_name='Средний охват'),
    x='Категория', y='Средний охват', color='Формат',
    title='Средний охват по форматам',
)
fig_reach.update_layout(xaxis_tickangle=-35)

format_perf.sort_values('Интегральный потенциал', ascending=False).head(10)
display(fig_formats)
display(fig_reach)
display(format_perf.sort_values('Интегральный потенциал', ascending=False).head(10))

,Категория,ER постов,ER клипов,Охват постов 30д,Охват клипов 30д,Интегральный потенциал
0,ANIMALS,0.081706,0.032046,6.289990e+05,237219.907301,0.646174
29,ENTERTAINMENT,0.041464,0.017077,1.686484e+06,606741.829760,0.587574
27,CULTURE_SOCIETY,0.053070,0.022479,9.524741e+05,147079.369552,0.572378
5,COMMUNITY,0.055448,0.029616,5.374542e+05,108167.644830,0.565744
17,COMPANY_MASS_MEDIA,0.024304,0.015160,2.281399e+06,247642.749913,0.559335
32,FOOD_COOKING,0.024325,0.010488,2.310947e+06,660274.986060,0.552387
22,COMPANY_SOCIETY,0.038897,0.018381,2.205031e+05,8764.944952,0.550367
30,FAMILY,0.042142,0.012586,9.973441e+05,293677.992052,0.544326
10,COMPANY_CULTURE,0.045946,0.023327,1.739044e+05,9832.848162,0.540371
11,COMPANY_EDUCATION,0.042080,0.030259,1.600644e+05,21447.739621,0.539675



## Сегменты × категории: где максимальный потенциал
Матрица поможет увидеть, какие комбинации сегментов и товарных ниш дают лучшую вовлечённость и охват.


In [6]:
segment_category = (
    df.groupby(['Сегмент', 'Категория'])
      .agg({
          'Интегральный потенциал': 'mean',
          'ER суммарный': 'mean',
          'Охват суммарный 30д': 'mean',
          'ID автора': 'count'
      })
      .rename(columns={'ID автора': 'Число авторов'})
      .reset_index()
)

# Топ-12 сочетаний по потенциалу
best_pairs = segment_category.sort_values('Интегральный потенциал', ascending=False).head(12)

fig_heat = px.density_heatmap(
    segment_category,
    x='Категория', y='Сегмент', z='Интегральный потенциал',
    color_continuous_scale=px.colors.sequential.Blues,
    title='Средний потенциал по сочетанию сегмент × категория'
)
fig_heat.update_layout(xaxis_tickangle=-35)

best_pairs
display(fig_heat)
display(best_pairs)

,Сегмент,Категория,Интегральный потенциал,ER суммарный,Охват суммарный 30д,Число авторов
160,5,COMPANY_ENTERTAINMENT,0.802899,0.053744,1.662797e+06,1
157,5,COMMUNITY,0.752546,0.085257,5.947727e+05,7
0,1,ANIMALS,0.742888,0.041398,1.932840e+06,7
158,5,COMPANY_CELEBRITY,0.733184,0.042908,4.376100e+05,3
163,5,COMPANY_MASS_MEDIA,0.728548,0.016316,1.417242e+06,2
27,1,TOURISM,0.707183,0.012986,7.150420e+06,3
3,1,CITIES,0.690041,0.014284,5.272587e+06,18
15,1,COMPUTER_IT,0.685117,0.029962,3.058680e+06,7
28,2,ANIMALS,0.684791,0.062724,2.321477e+06,1224
162,5,COMPANY_INTERNET,0.681697,0.059512,3.656880e+05,1



## Выводы и рекомендации
- **Сегменты с максимальным потенциалом.** Лидируют сегменты с высоким средним `Интегральный потенциал` и достаточным числом авторов (см. график по сегментам). Их стоит приоритизировать в пилотах соцкоммерции.
- **Категории с наивысшим ER и потенциалом.** Топ-товарные категории (см. пузырьковую диаграмму) объединяют высокий `ER суммарный` и значимую базу авторов — здесь проще быстро получить продажи.
- **Форматы.** Сравнение `ER постов` и `ER клипов` показывает, какие ниши лучше работают через короткие клипы, а какие — через посты. Для категорий, где клипы доминируют, рекомендованы динамичные обзоры и демонстрации товара; для постов — развёрнутые гайды и подборки.
- **Топ-авторы.** Таблица топ-15 отражает авторов с наибольшим потенциалом: у них высокие охваты, регулярный выпуск контента и выше средняя вовлечённость. Их можно использовать как якорных партнёров.
- **Сочетания сегмент × категория.** Тепловая карта подсвечивает сочетания, где одновременно высокий потенциал и достаточно авторов: на них стоит запускать эксперименты с коммерческими интеграциями в первую очередь.
